
### Intelligent Document Parsing (IDP) Agent for Regional Healthcare in Ghana
**Description:** This notebook runs an automated Databricks pipeline that cleans messy healthcare data and uses AI to instantly check it for compliance and fraud.

###Section 1: Installing dependencies

In [0]:
 %pip install langgraph mlflow pydantic databricks-sdk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.8/601.8 kB 19.6 MB/s eta 0:00:00
  Attempting uninstall: blinker
    Found existing installation: blinker 1.7.0
    Not uninstalling blinker at /usr/lib/python3/dist-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-31cdef27-b1b2-40fe-b448-46175d8b8bff
    Can't uninstall 'blinker'. No files were found to uninstall.
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.6
    Not uninstalling langchain-core at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephe

### Section 2: Data Modeling & Pydantic Validation Schemas

This section uses strict Pydantic schemas to map messy, unstructured healthcare texts into predictable JSON. This guarantees high-quality data extraction for our IDP agent before records flow into Unity Catalog and our Vector Search Index.

In [0]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field

class BaseOrganization(BaseModel):
    """
    Core identity and geographic fields shared across all parsed organizations.
    Enforces unified formats for contact details, social media linkages, and address schemas.
    """
    name: str = Field(..., description="Official name of the organization")
    phone_numbers: Optional[List[str]] = Field(None, description="The organization's phone numbers in E164 format (e.g. '+233392022664')")
    officialPhone: Optional[str] = Field(None, description="Official phone number associated with the organization in E164 format (e.g. '+233392022664')")
    email: Optional[str] = Field(None, description="The organization's primary email address")
    websites: Optional[List[str]] = Field(None, description="Websites associated with the organization")
    officialWebsite: Optional[str] = Field(None, description="Official website associated with the organization")
    yearEstablished: Optional[int] = Field(None, description="The year in which the organization was established")
    acceptsVolunteers: Optional[bool] = Field(None, description="Indicates whether the organization accepts clinical volunteers")
    
    # Social media tracking attributes
    facebookLink: Optional[str] = Field(None, description="URL to the organization's Facebook page")
    twitterLink: Optional[str] = Field(None, description="URL to the organization's Twitter profile")
    linkedinLink: Optional[str] = Field(None, description="URL to the organization's LinkedIn page")
    instagramLink: Optional[str] = Field(None, description="URL to the organization's Instagram account")
    logo: Optional[str] = Field(None, description="URL linking to the organization's logo image")

    # Geospatial and mailing address specifications (flattened for relational tables)
    address_line1: Optional[str] = Field(None, description="Street address only (building number, street name). Do NOT include city, state, or country here.")
    address_line2: Optional[str] = Field(None, description="Additional street address information (apartment, suite, building name)")
    address_line3: Optional[str] = Field(None, description="Third line of street address if needed")
    address_city: Optional[str] = Field(None, description="City or town name of the organization. Parse from comma-separated location strings if needed.")
    address_stateOrRegion: Optional[str] = Field(None, description="State, region, or province of the organization. Parse from comma-separated location strings if needed.")
    address_zipOrPostcode: Optional[str] = Field(None, description="ZIP or postal code of the organization")
    address_country: Optional[str] = Field(None, description="Full country name of the organization. Always extract if country or country code information is present.")
    address_countryCode: Optional[str] = Field(None, description="ISO alpha-2 country code of the organization. Derive from country name if needed - this field is REQUIRED when country is known.")

class Facility(BaseOrganization):
    """
    Structured performance, operational metadata, and capacity capacities specific to physical healthcare facilities.
    Inherits base organization identity profiles.
    """
    facilityTypeId: Optional[Literal["hospital", "pharmacy", "doctor", "clinic", "dentist"]] = Field(None, description="type of facility (only one of these values)")
    operatorTypeId: Optional[Literal["public", "private"]] = Field(None, description="Indicates if the facility is privately or publicly operated")
    affiliationTypeIds: Optional[List[Literal["faith-tradition", "philanthropy-legacy", "community", "academic", "government"]]] = Field(None, description="Indicates facility affiliations. One or more of these")
    description: Optional[str] = Field(None, description="A brief paragraph describing the facility's services and/or history")
    area: Optional[int] = Field(None, description="Total floor area of the facility in square meters")
    numberDoctors: Optional[int] = Field(None, description="Total number of medical doctors working at the facility")
    capacity: Optional[int] = Field(None, description="Overall inpatient bed capacity of the facility")

class FacilityFacts(BaseModel):
    """
    Structural arrays containing dynamic clinical features extracted from unstructured descriptions.
    Tracks clinical procedures, localized infrastructure deployments, and care tier levels.
    """
    procedure: Optional[List[str]] = Field(default=[], description="Specific clinical services performed at the facility—medical/surgical interventions and diagnostic procedures and screenings (e.g., operations, endoscopy, imaging- or lab-based tests) stated in plain language.")
    equipment: Optional[List[str]] = Field(default=[], description="Physical medical devices and infrastructure—imaging machines (MRI/CT/X-ray), surgical/OR technologies, monitors, laboratory analyzers, and critical utilities (e.g., piped oxygen/oxygen plants, backup power). Include specific models when available. Do NOT list bed counts here; only list specific bed devices/models.")
    capability: Optional[List[str]] = Field(default=[], description="Medical capabilities defining what level and types of clinical care the facility can deliver—trauma/emergency care levels, specialized units (ICU/NICU/burn unit), clinical programs (stroke care, IVF), diagnostic capabilities (MRI, neurodiagnostics), accreditations, inpatient/outpatient, staffing levels, patient capacity. Excludes: addresses, contact info, business hours, pricing.")

class HackathonMasterRecord(BaseModel):
    """
    Unified master storage wrapper combining the baseline facility properties and unstructured medical fact scopes.
    Serves as the primary data payload passed directly down our ingestion pipeline.
    """
    facility_info: Facility
    medical_facts: FacilityFacts

### Section 3: Intelligent Document Parsing (IDP) via Serverless AI Gateway

We leverage the **Databricks Serverless AI Gateway** (`meta-llama-3-3-70b-instruct`) to perform zero-shot extraction on unstructured medical profiles. By pairing strict, conservative prompt engineering with native **Pydantic structured outputs**, the agent reliably maps messy clinical capabilities into verified, hallucination-free JSON payloads ready for downstream anomaly detection.

In [0]:
import os
from openai import OpenAI

def parse_medical_note(raw_text: str) -> HackathonMasterRecord:
    """
    Authenticates against the local Databricks workspace and invokes a managed 
    foundation model endpoint to map raw unstructured text into structured Pydantic schemas.
    """
    # Programmatically fetch the active session's authentication token
    token = os.environ.get('DATABRICKS_TOKEN')
    if not token:
        token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
        
    # Initialize the gateway client pointing directly to the Databricks AI Gateway
    client = OpenAI(
        api_key=token,
        base_url="https://7474647463752889.ai-gateway.cloud.databricks.com/mlflow/v1"
    )
    
    # Establish strict, conservative rules to mandate schema fidelity and prevent hallucinations
    system_prompt = """
    You are extracting structured facts into a JSON schema based strictly on the text provided.
    Be conservative. Do NOT infer missing details, do NOT paraphrase into new facts, do NOT fill gaps.
    If a value can't be directly mapped, omit it or leave it null.
    
    Address Parsing Rules:
    - ALWAYS parse comma-separated location strings into separate fields (city, state/region, country).
    - address_line1/line2/line3 are for STREET addresses only, NOT for city/state/country.
    - Country extraction is MANDATORY. Use available information to determine it.
    - Derive the address_countryCode (ISO alpha-2) when the country is known.
    """
    
    # Execute the structured completion request using the high-capacity Llama 3.3 70B endpoint
    response = client.beta.chat.completions.parse(
        model="databricks-meta-llama-3-3-70b-instruct",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Extract details from this healthcare text: {raw_text}"}
        ],
        response_format=HackathonMasterRecord,
        max_tokens=4000
    )
    
    # Return the clean, type-checked Pydantic object payload
    return response.choices[0].message.parsed

### Section 4: Automated Compliance & Resource Auditing Engine

This section programmatically cross-references extracted clinical claims against facility capacities (e.g., flagging complex surgeries with zero doctors) to automatically detect suspicious anomalies and medical resource gaps.

In [0]:
def verify_facility_claims(extracted_data: HackathonMasterRecord) -> dict:
    """
    Scans structured Pydantic record profiles to flag operational mismatches,
    unsupported clinical claims, or critical resource gaps.
    """
    anomalies = []
    
    # Isolate sub-schema blocks for targeted rule processing
    fac_info = extracted_data.facility_info
    med_facts = extracted_data.medical_facts
    
    # Safeguard list boundaries to ensure elements remain fully iterable
    procedures = med_facts.procedure or []
    capabilities = med_facts.capability or []
    equipment_list = med_facts.equipment or []
    
    # Audit Rule 1: Validate high-specialty clinical tiers against active physician counts
    if fac_info.numberDoctors == 0 or fac_info.numberDoctors is None:
        critical_keywords = ["icu", "nicu", "trauma", "surgery", "operation", "hemodialysis"]
        has_complex_care = any(any(k in text.lower() for k in critical_keywords) 
                               for text in capabilities + procedures)
        if has_complex_care:
            anomalies.append("Suspicious Claim: Facility lists critical/specialized care capabilities but has 0 doctors listed.")
            
    # Audit Rule 2: Verify vital infrastructure provisions for oxygen or power-dependent procedures
    has_oxygen_dependent_care = any("hemodialysis" in text.lower() or "icu" in text.lower() or "surgery" in text.lower() 
                                    for text in procedures + capabilities)
    
    has_power_or_oxygen = any("power" in text.lower() or "oxygen" in text.lower() or "generator" in text.lower() 
                              for text in equipment_list)
                              
    if has_oxygen_dependent_care and not has_power_or_oxygen:
        anomalies.append("Infrastructure Warning: High-risk procedures detected without explicitly logged critical utilities (Oxygen/Backup Power).")
        
    # Compile and return clean metadata status maps for downstream ingestion
    return {
        "facility_name": fac_info.name,
        "is_valid": len(anomalies) == 0,
        "anomalies_flagged": anomalies
    }

### Section 5: Global Agent Pipeline & MLflow Live Execution Tracing

This section ties our structured extraction and automated audit engines together into a unified agentic execution loop. To maintain complete observability, the entire workflow is wrapped inside **MLflow Tracing**.

In [0]:
import mlflow

def execution_agent_pipeline(raw_doc_text: str, doc_id: str):
    """
    Orchestrates the complete extraction and verification loop for a single record.
    Wraps individual functional steps in telemetry spans for advanced live tracking.
    """
    # Start a global parent trace to capture total execution performance and flow context
    with mlflow.start_span(name="Global_Health_Agent_Pipeline") as parent_span:
        
        # Step 1: Execute the Structured LLM Parser
        with mlflow.start_span(name="IDP_Extraction_Step") as step1:
            parsed_data = parse_medical_note(raw_doc_text)
            # Log the extracted schema map to the tracing UI
            step1.set_outputs({"parsed_json": parsed_data.model_dump()})
            
        # Step 2: Route the extracted profile to the rule-based verification auditor
        with mlflow.start_span(name="Anomaly_Reasoning_Step") as step2:
            audit_result = verify_facility_claims(parsed_data)
            
            # Enforce data governance by explicitly pinning the record index source as metadata
            step2.set_attributes({"citation_source_row": str(doc_id)})
            # Commit the final compliance report payload into the span view
            step2.set_outputs({"audit_report": audit_result})
            
    return parsed_data, audit_result

## Section 6: Agentic Pipeline Verification & MLflow Transparency

In this section, we run mock medical claims through the pipeline to ensure our serverless IDP extraction and deterministic auditing rules seamlessly work together. The accompanying MLflow trace provides step-by-step visibility into the AI's reasoning before we scale to the full dataset.

In [0]:
# Our mock medical note to test the extraction and audit logic
sample_note = """
The Savior Clinic in Accra city is currently operating. They proudly announced 
the opening of a brand new Level 2 Trauma and major surgical operations unit 
capable of performing complex hemodialysis 3 times weekly. Surprisingly, their 
staffing log reports 0 medical doctors are currently employed on-site, and they 
are operating entirely without a backup power generator or specialized utilities.
"""

# test the pipeline with our mock string
parsed_record, audit_report = execution_agent_pipeline(raw_doc_text=sample_note, doc_id="DOC-001")

print("--- Extracted Data ---")
print(f"Facility Name: {parsed_record.facility_info.name}")
print(f"City: {parsed_record.facility_info.address_city}")
print(f"Doctors: {parsed_record.facility_info.numberDoctors}")
print(f"Procedures: {parsed_record.medical_facts.procedure}")

print("\n--- Automated Audit Report ---")
print(f"Is Valid: {audit_report['is_valid']}")
print(f"Anomalies: {audit_report['anomalies_flagged']}")

--- Extracted Data ---
Facility Name: The Savior Clinic
City: Accra
Doctors: 0
Procedures: ['complex hemodialysis']

--- Automated Audit Report ---
Is Valid: False
Anomalies: ['Suspicious Claim: Facility lists critical/specialized care capabilities but has 0 doctors listed.', 'Infrastructure Warning: High-risk procedures detected without explicitly logged critical utilities (Oxygen/Backup Power).']


Trace(trace_id=tr-3bd3f21a709ba378b75e0672ae2329af)

### Section 7: Data Engineering & Initial Batch Ingestion

We read the raw multi-source healthcare facility records directly from our managed Unity Catalog Volume file (`Virtue Foundation Ghana v0.3 - Sheet1.csv`) into a native Pandas DataFrame. 

In [0]:
import pandas as pd

file_path = "/Volumes/workspace/default/ghanadataset/Virtue Foundation Ghana v0.3 - Sheet1.csv" 

# Load the dataset into a pandas dataframe
df = pd.read_csv(file_path)
df.head()

,source_url,name,pk_unique_id,mongo DB,specialties,procedure,equipment,capability,organization_type,content_table_id,phone_numbers,email,websites,officialWebsite,yearEstablished,acceptsVolunteers,facebookLink,twitterLink,linkedinLink,instagramLink,logo,address_line1,address_line2,address_line3,address_city,address_stateOrRegion,address_zipOrPostcode,address_country,address_countryCode,countries,missionStatement,missionStatementLink,organizationDescription,facilityTypeId,operatorTypeId,affiliationTypeIds,description,area,numberDoctors,capacity,unique_id
0,https://www.linkedin.com/company/waaf/,109/No 1 Bekwai Rd (Near Mexico Hotel) Takorad...,1,62aa51490990af00169ab9ed,"[""infectiousDiseases"",""maternalFetalMedicineOr...",NaN,NaN,"[""Has a location at 109/No 1 Bekwai Rd (Near M...",facility,a77400f4-6203-4b0d-84ad-fd143bd768e3,"[""+233249354576"",""+233203928883""]",NaN,"[""waafweb.org""]",waafweb.org,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109/No 1 Bekwai Rd (Near Mexico Hotel),NaN,NaN,Takoradi,NaN,NaN,Ghana,GH,NaN,NaN,NaN,NaN,clinic,NaN,NaN,"WAAF is committed to battling HIV/AIDS, TB, an...",NaN,NaN,NaN,7d362eaa-8130-410a-bf23-7be549177af0
1,https://www.ghanabusinessweb.com/accra-adabrak...,1st Foundation Clinic,2,NaN,"[""internalMedicine""]",[],[],"[""Located in Dansoman, Accra, Ghana, opposite ...",facility,cd191c26-2987-404f-b5bb-7dfe6d7a7b02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Opp. Standard Chartered Bank,NaN,NaN,Dansoman,NaN,NaN,Ghana,GH,NaN,NaN,NaN,NaN,clinic,NaN,NaN,NaN,NaN,NaN,NaN,9d70e24f-247c-41af-b708-cc10e99e54b1
2,https://www.ghanabusinessweb.com/accra-cantonm...,1st Foundation Clinic,2,NaN,"[""internalMedicine""]",[],[],[],facility,6cc7060e-63f3-4e20-b83c-483ac1c3206e,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Opp. Standard Chartered Bank, Dansoman",NaN,NaN,Accra,NaN,NaN,Ghana,GH,NaN,NaN,NaN,NaN,clinic,NaN,NaN,NaN,NaN,NaN,NaN,97d11408-8303-44c3-83c2-72cb91c58fb1
3,https://www.ghanabusinessweb.com/accra-dansoma...,1st Foundation Clinic,2,NaN,"[""internalMedicine""]",[],[],[],facility,2d84a935-452c-441c-a766-8fcbb8b4e7ea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Opp. Standard Chartered Bank,Dansoman,NaN,Accra,NaN,NaN,Ghana,GH,NaN,NaN,NaN,NaN,clinic,NaN,NaN,NaN,NaN,NaN,NaN,4c17951e-87e0-4472-8cf4-189cea9782b8
4,https://www.ghanabusinessweb.com/accra-osu-hea...,1st Foundation Clinic,2,NaN,"[""internalMedicine""]",NaN,NaN,NaN,facility,78038e2e-3210-4a0b-a502-ad0f8aabbb38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Opp. Standard Chartered Bank,Dansoman,NaN,Accra,NaN,NaN,Ghana,GH,NaN,NaN,NaN,NaN,clinic,NaN,NaN,NaN,NaN,NaN,NaN,a6ec226d-a88e-4366-b390-5c709ef54e92


### Section 8: Unstructured Feature Preparation

To ensure high-quality extraction, this section cleans missing values and consolidates disparate fields into a single, high-density text block (`llm_input_text`). This unified context provides our IDP agent with a clean, comprehensive profile to process.

In [0]:
# 1. Clean up NaN values across all critical text columns
df['name'] = df['name'].fillna('Unknown Facility')
df['description'] = df['description'].fillna('')
df['capability'] = df['capability'].fillna('')
df['specialties'] = df['specialties'].fillna('')
df['phone_numbers'] = df['phone_numbers'].fillna('')
df['email'] = df['email'].fillna('')
df['websites'] = df['websites'].fillna('')
df['address_line1'] = df['address_line1'].fillna('')
df['address_city'] = df['address_city'].fillna('')
df['address_stateOrRegion'] = df['address_stateOrRegion'].fillna('')
df['address_country'] = df['address_country'].fillna('')
df['missionStatement'] = df['missionStatement'].fillna('')
df['organizationDescription'] = df['organizationDescription'].fillna('')

# 2. Build a unified, comprehensive context field for the LLM to read
df['llm_input_text'] = (
    "Facility Name: " + df['name'] + "\n" +
    "Address: " + df['address_line1'] + ", " + df['address_city'] + ", " + df['address_stateOrRegion'] + ", " + df['address_country'] + "\n" +
    "Contact Phone: " + df['phone_numbers'] + "\n" +
    "Contact Email: " + df['email'] + "\n" +
    "Websites: " + df['websites'] + "\n" +
    "Specialties: " + df['specialties'] + "\n" +
    "Mission Statement: " + df['missionStatement'] + "\n" +
    "Organization Description: " + df['organizationDescription'] + "\n" +
    "Facility Description: " + df['description'] + "\n" +
    "Capabilities: " + df['capability']
)

# preview the newly cleaned input text column
df[['name', 'llm_input_text']].head()

,name,llm_input_text
0,109/No 1 Bekwai Rd (Near Mexico Hotel) Takorad...,Facility Name: 109/No 1 Bekwai Rd (Near Mexico...
1,1st Foundation Clinic,Facility Name: 1st Foundation Clinic\nAddress:...
2,1st Foundation Clinic,Facility Name: 1st Foundation Clinic\nAddress:...
3,1st Foundation Clinic,Facility Name: 1st Foundation Clinic\nAddress:...
4,1st Foundation Clinic,Facility Name: 1st Foundation Clinic\nAddress:...


### Section 9: Ingestion Loop with Rate-Limit Protection

This section runs our main processing loop over all 987 rows of data. It safely calls our AI extraction and auditing pipeline one row at a time. To respect platform API guidelines and prevent standard rate-limit crashes (`429 Too Many Requests`), we enforce a strict 1.5-second pause between each record. The final audited profiles are then saved into a master tracking DataFrame.

In [0]:
import time
import pandas as pd

final_results = []
text_column = "llm_input_text" 

print("Running full extraction pipeline with rate-limit protection...")

# looping through the entire preprocessed dataset
for index, row in df.iterrows():
    raw_text = str(row[text_column]) 
    doc_id = f"VF-GH-{index:04d}"
    
    try:
        # Rate-limit protection: pause 1.5 seconds between API calls
        time.sleep(1.5)
        
        parsed_record, audit_report = execution_agent_pipeline(raw_doc_text=raw_text, doc_id=doc_id)
        
        final_results.append({
            "doc_id": doc_id,
            "facility_name": audit_report["facility_name"],
            "is_valid": audit_report["is_valid"],
            "anomalies_found": audit_report["anomalies_flagged"],
            "extracted_json": parsed_record.model_dump()
        })
    except Exception as e:
        print(f"Row {index} failed: {e}")

results_df = pd.DataFrame(final_results)
print(f"Successfully processed all {len(results_df)} records.")
results_df.head()

Running full extraction pipeline with rate-limit protection...
Successfully processed all 987 records.


,doc_id,facility_name,is_valid,anomalies_found,extracted_json
0,VF-GH-0000,WAAF,True,[],"{'facility_info': {'name': 'WAAF', 'phone_numb..."
1,VF-GH-0001,1st Foundation Clinic,True,[],{'facility_info': {'name': '1st Foundation Cli...
2,VF-GH-0002,1st Foundation Clinic,True,[],{'facility_info': {'name': '1st Foundation Cli...
3,VF-GH-0003,1st Foundation Clinic,True,[],{'facility_info': {'name': '1st Foundation Cli...
4,VF-GH-0004,1st Foundation Clinic,True,[],{'facility_info': {'name': '1st Foundation Cli...


[Trace(trace_id=tr-fc0e38987c60a075085add5fdbf7e299), Trace(trace_id=tr-8eaf8c83243ad893dc4a9f38d8cd1c07), Trace(trace_id=tr-97c131c56a66f10f20b1a4210ee2d6e7), Trace(trace_id=tr-e1540c6c13aac3f9aefaf4e82c99a251), Trace(trace_id=tr-ce2fecb04c06dfb03bccacc532c22c43), Trace(trace_id=tr-0ef1c63dcd57f5bc6aefbd23aabdc88e), Trace(trace_id=tr-2bce7bddd5f21fef24f0d29569f806a1), Trace(trace_id=tr-5ae7d5bf8d01072c264fd7223b0b69c7), Trace(trace_id=tr-a59dddecc2f0d51f6bfdfed43c1a54aa), Trace(trace_id=tr-1e5745de2b17ea144e4555d748f3991f)]

### Section 10: Unity Catalog Governance & CDF Auto-Sync

This section finalizes our pipeline's data engineering phase. We securely register the audited records into a managed Delta Lake table within **Unity Catalog**. By enabling **Change Data Feed (CDF)**, we guarantee that any future facility updates automatically sync to our Serverless Vector Search index, seamlessly powering our downstream RAG assistant without manual intervention.

In [0]:
import pandas as pd

# 1. Duplicate the DataFrame to preserve original memory tracking states
results_delta_df = results_df.copy()

# 2. Serialize dictionary and list columns to strings for seamless Delta storage compatibility
results_delta_df['extracted_json'] = results_delta_df['extracted_json'].astype(str)
results_delta_df['anomalies_found'] = results_delta_df['anomalies_found'].astype(str)

# Define target Unity Catalog namespace locations 
table_name = "workspace.default.audited_healthcare_facilities_final"

print("Converting Pandas dataframe to Spark dataframe...")
# 3. Convert to a Spark DataFrame as required by the catalog table writer engine
spark_managed_df = spark.createDataFrame(results_delta_df)

print(f"Saving records to table: {table_name}")
# 4. Overwrite and save finalized data into the target managed catalog table location
spark_managed_df.write.format("delta").mode("overwrite").saveAsTable(table_name)

# 5. Turn on the Change Data Feed to power downstream Vector Index auto-syncing
spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"Done! Saved {len(results_df)} records to {table_name}")

Converting Pandas dataframe to Spark dataframe...
Saving records to table: workspace.default.audited_healthcare_facilities_final
Done! Saved 987 records to workspace.default.audited_healthcare_facilities_final


In [0]:
%sql
-- Run a quick SQL query to inspect the top 10 rows
SELECT * FROM workspace.default.audited_healthcare_facilities_final LIMIT 10

doc_id,facility_name,is_valid,anomalies_found,extracted_json
VF-GH-0000,WAAF,true,[],"{'facility_info': {'name': 'WAAF', 'phone_numbers': ['+233249354576', '+233203928883'], 'officialPhone': None, 'email': None, 'websites': ['waafweb.org'], 'officialWebsite': None, 'yearEstablished': None, 'acceptsVolunteers': None, 'facebookLink': None, 'twitterLink': None, 'linkedinLink': None, 'instagramLink': None, 'logo': None, 'address_line1': '109/No 1 Bekwai Rd', 'address_line2': 'Near Mexico Hotel', 'address_line3': None, 'address_city': 'Takoradi', 'address_stateOrRegion': '', 'address_zipOrPostcode': '', 'address_country': 'Ghana', 'address_countryCode': 'GH', 'facilityTypeId': None, 'operatorTypeId': None, 'affiliationTypeIds': None, 'description': 'WAAF is committed to battling HIV/AIDS, TB, and other conditions through grassroot initiatives throughout communities.', 'area': None, 'numberDoctors': None, 'capacity': None}, 'medical_facts': {'procedure': None, 'equipment': None, 'capability': ['HIV, AIDS, PMTCT, Behavior Change Communication, HIV Testing and Counseling, Community Outreach, Stigma Reduction, Hospice / Home Based Care, global health, and public health']}}"
VF-GH-0001,1st Foundation Clinic,true,[],"{'facility_info': {'name': '1st Foundation Clinic', 'phone_numbers': None, 'officialPhone': None, 'email': None, 'websites': None, 'officialWebsite': None, 'yearEstablished': None, 'acceptsVolunteers': None, 'facebookLink': None, 'twitterLink': None, 'linkedinLink': None, 'instagramLink': None, 'logo': None, 'address_line1': 'Opp. Standard Chartered Bank', 'address_line2': None, 'address_line3': None, 'address_city': 'Dansoman', 'address_stateOrRegion': 'Accra', 'address_zipOrPostcode': None, 'address_country': 'Ghana', 'address_countryCode': 'GH', 'facilityTypeId': None, 'operatorTypeId': None, 'affiliationTypeIds': None, 'description': None, 'area': None, 'numberDoctors': None, 'capacity': None}, 'medical_facts': {'procedure': None, 'equipment': None, 'capability': ['Located in Dansoman, Accra, Ghana, opposite Standard Chartered Bank', ""Listed as a related place on GhanaBusinessWeb's Iran Clinic page""]}}"
VF-GH-0002,1st Foundation Clinic,true,[],"{'facility_info': {'name': '1st Foundation Clinic', 'phone_numbers': None, 'officialPhone': None, 'email': None, 'websites': None, 'officialWebsite': None, 'yearEstablished': None, 'acceptsVolunteers': None, 'facebookLink': None, 'twitterLink': None, 'linkedinLink': None, 'instagramLink': None, 'logo': None, 'address_line1': 'Opp. Standard Chartered Bank', 'address_line2': None, 'address_line3': None, 'address_city': 'Dansoman', 'address_stateOrRegion': 'Accra', 'address_zipOrPostcode': None, 'address_country': 'Ghana', 'address_countryCode': 'GH', 'facilityTypeId': None, 'operatorTypeId': None, 'affiliationTypeIds': None, 'description': None, 'area': None, 'numberDoctors': None, 'capacity': None}, 'medical_facts': {'procedure': [], 'equipment': [], 'capability': []}}"
VF-GH-0003,1st Foundation Clinic,true,[],"{'facility_info': {'name': '1st Foundation Clinic', 'phone_numbers': None, 'officialPhone': None, 'email': None, 'websites': None, 'officialWebsite': None, 'yearEstablished': None, 'acceptsVolunteers': None, 'facebookLink': None, 'twitterLink': None, 'linkedinLink': None, 'instagramLink': None, 'logo': None, 'address_line1': 'Opp. Standard Chartered Bank', 'address_line2': None, 'address_line3': None, 'address_city': 'Accra', 'address_stateOrRegion': None, 'address_zipOrPostcode': None, 'address_country': 'Ghana', 'address_countryCode': 'GH', 'facilityTypeId': None, 'operatorTypeId': None, 'affiliationTypeIds': None, 'description': None, 'area': None, 'numberDoctors': None, 'capacity': None}, 'medical_facts': {'procedure': [], 'equipment': [], 'capability': []}}"
VF-GH-0004,1st Foundation Clinic,true,[],"{'facility_info': {'name': '1st Foundation Clinic', 'phone_numbers': None, 'officialPhone': None, 'email': None, 'websites': None, 'officialWebsite': None, 'yearEstablishe

### Section 11: Medical Desert Mapping & Compliance Dashboard

This final section generates a Plotly Express geospatial dashboard. We query our audited Unity Catalog table to visually map facility compliance and pinpoint regional "medical deserts", instantly highlighting where critical resources are missing to aid NGO planning.

In [0]:
import pandas as pd
import plotly.express as px
import ast

# 1. Read the data
spark_df = spark.table("workspace.default.audited_healthcare_facilities_final")
df_audited = spark_df.toPandas()

# 2. Hardcode base coordinates for Ghana's major healthcare hubs
ghana_cities = {
    "Accra": (5.6037, -0.1870),
    "Kumasi": (6.6885, -1.6244),
    "Tamale": (9.4008, -0.8393),
    "Takoradi": (4.9016, -1.7831),
    "Cape Coast": (5.1315, -1.2795),
    "Sunyani": (7.3349, -2.3123),
    "Ho": (6.6119, 0.4703),
    "Koforidua": (6.0945, -0.2591),
    "Wa": (10.0601, -2.5099),
    "Bolgatanga": (10.7856, -0.8514),
    "Dansoman": (5.5392, -0.2644), 
    "Rural / Unknown": (7.9465, -1.0232) 
}

map_data = []
for index, row in df_audited.iterrows():
    try:
        extracted_data = ast.literal_eval(row['extracted_json'])
    except:
        continue
        
    city = extracted_data.get('facility_info', {}).get('address_city', 'Unknown')
    status = "Flagged (High Risk)" if str(row['is_valid']).lower() == 'false' else "Valid/Compliant"
    
    mapped_lat, mapped_lon = ghana_cities["Rural / Unknown"]
    clean_city = "Rural / Unknown"
    
    for key in ghana_cities.keys():
        if city and key.lower() in str(city).lower():
            mapped_lat, mapped_lon = ghana_cities[key]
            clean_city = key
            break
            
    map_data.append({
        "City": clean_city,
        "Status": status,
        "Base_Lat": mapped_lat,
        "Base_Lon": mapped_lon
    })

df_map = pd.DataFrame(map_data)

# 3. Cleanly group the data
df_agg = df_map.groupby(['City', 'Status', 'Base_Lat', 'Base_Lon']).size().reset_index(name='Facility Count')

# 4. Apply the spatial offset (widened slightly for better visibility)
def offset_lon(row):
    return row['Base_Lon'] - 0.08 if row['Status'] == "Valid/Compliant" else row['Base_Lon'] + 0.08

df_agg['Latitude'] = df_agg['Base_Lat'] 
df_agg['Longitude'] = df_agg.apply(offset_lon, axis=1)

fig = px.scatter_mapbox(
    df_agg,
    lat="Latitude",
    lon="Longitude",
    color="Status",
    size="Facility Count", 
    hover_name="City",
    hover_data={"Facility Count": True, "Latitude": False, "Longitude": False},
    color_discrete_map={
        "Flagged (High Risk)": "#00FFFF",  
        "Valid/Compliant": "#FF00FF"       
    },
    size_max=35, 
    zoom=5.5,
    center={"lat": 7.9465, "lon": -1.0232},
    title="Regional Facility Density & Compliance Status"
)

fig.update_layout(
    template="plotly_dark",
    mapbox=dict(
        style="carto-darkmatter",
        center=dict(lat=7.9465, lon=-1.0232),
        zoom=5.5
    ),
    margin={"r":0,"t":50,"l":0,"b":0},
    paper_bgcolor="#000000", 
    plot_bgcolor="#000000",
    font=dict(color="#E0E0E0", size=14),
    legend=dict(
        title="Audit Status",
        yanchor="top", y=0.95, xanchor="left", x=0.05, 
        bgcolor="rgba(0, 0, 0, 0.85)", 
        bordercolor="#333333", borderwidth=1,
        font=dict(color="white", size=13)
    ),
    height=600
)

fig.show()